<a href="https://colab.research.google.com/github/jruttan1/ComancheNLP/blob/main/algoverse_takehome.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# A

1.
The experiments on DistilBERT in results.md conclude that InfoNCE was the best method for measuring layer similarity to determine how to add MITR. When expanding the experiment to BERT and RoBERTa shown in the roberta_bert_results.md file, larger 12-layer models, they did not test InfoNCE, the best-performing strategy from DistilBERT experiments. They tested cosine similarity and CKA, which both proved less effective when used on DistilBERT for the same purposes. This is justified in the write-up by saying "We skipped CLUB (diverges completely on DistilBERT) and InfoNCE (scoped out for this round)". Excluding CLUB makes sense given it diverged completely on DistilBERT, with its loss spiking to the -50 clamp by epoch 2, rendering it useless for larger models. But InfoNCE, which was concluded as the best overall method, was excluded for being out of scope which is not a valid reason to cut out a likely effective method of testing your hypothesis. A reviewer would reject this claim because you conclude with preliminary experiments that a method (InfoNCE) is the most effective at accomplishing your goal of reducing contradiction rate, and they exclude it from your experiments on more robust models without justification, making it impossible to know if the paper's central finding holds on more capable models, particularly given that InfoNCE's +0.20% accuracy gain on DistilBERT is too small to be meaningful without repeated runs and variance estimates.

To address this weakness, run InfoNCE on BERT and RoBERTa using the existing mitr_bert_roberta_boolq.ipynb notebook and compare results against Cosine and CKA on both accuracy and contradiction rate. Each configuration should also be run at minimum 3 times with different random seeds to establish statistical significance given the small accuracy differences that were observed in the DistilBERT exeperiments.

2.
A second critical weakness is the complete contradiction in outcomes between BERT and RoBERTa without further investigation or explanation. These models are architecturally identical with variations coming from the pretraining recipe. A completely opposite effect from the same methods on two architecturally identical models cannot be presented as conclusive. When both the CKA and cosine methods are applied to BERT, they increase its fractional contradiction rate from 0.4160 to 0.4480 with cosine and 0.4560 with CKA (making the model less consistent at correctly answering BoolQ), in comparison to baseline. Whereas RoBERTa, with the same methods, has a lowered fractional contradiction rate compared to baseline going from 0.5400 to 0.5080 with cosine and 0.5140 with CKA. This means that the same methods applied to two architecturally identical models with different pretraining objectives improved one and worsened the other, which cannot be claimed as conclusive without further testing. A reviewer would reject this because the same MITR methods improve logical consistency on RoBERTa but worsen it on BERT, meaning the paper cannot make a general claim that MITR improves logical reasoning.

To fix this weakness, run the same MITR experiments with cosine and CKA on 2-3 more similar transformer-based models to conclude whether RoBERTa improving is an outlier or BERT worsening is the outlier. If the majority of additional backbones show improved contradiction rates, it would support MITR as a generally effective method with BERT as a pretraining-dependent exception.

# B

Workshop: BlackboxNLP 2025: The 8th Workshop on Analyzing and Interpreting Neural Networks for NLP, co-located with EMNLP 2025, Suzhou, China.

MITR directly addresses a core question in BlackboxNLP's scope: What computations do transformer layers actually perform, and can we engineer behavior of the layers to be more interpretable? By applying mutual information penalties to force distinct layer representations, MITR is a concrete architectural modification aimed at reducing redundancy between layers which is a direction listed in the workshop's topics of interest:  "proposing modifications to neural architectures that increase their interpretability". The backbone dependent results, where MITR improves consistency on RoBERTa but hurts it on BERT, raise questions about how pretraining objectives shape inter-layer redundancy, exactly the kind of mechanistic question BlackboxNLP targets.

Related papers that were published at this workshop are:

Mechanistic Fine-tuning for In-context Learning — Cho et al., BlackboxNLP 2025. Both papers build training objectives targeting intermediate layer behavior, ABFT on attention scores and MITR on inter-layer mutual information, to improve model performance without modifying final output supervision.

Investigating ReLoRA: Effects on the Learning Dynamics of Small Language Models — Weiss et al., BlackboxNLP 2025. Both papers study how modifying a training objective alters internal layer dynamics in transformer models, ReLoRA through low-rank adapter resets during pretraining and MITR through mutual information penalties during fine-tuning.

# C

1.
InfoNCE-BERT-RoBERTa.
Hypothesis: If we apply MITR-InfoNCE to BERT and RoBERTa, then both models will show a lower contradiction rate than their respective baselines (0.4160 and 0.5400), because InfoNCE's contrastive learning objective produces stronger inter-layer diversity signals than the parameter-free Cosine and CKA methods, which failed to improve BERT's logical consistency in the previous experiment.

Expected result: Both BERT and RoBERTa show contradiction rates below their respective baselines of 0.4160 and 0.5400.

What a negative result means: This hypothesis could be wrong in three distinctly different ways. If the findings show that MITR-InfoNCE reduces the contradiction rate for only RoBERTa, this strengthens the overall conclusion that MITR is ineffective for BERT. If it increases contradiction rate for both models, this shows that parameterized methods of measuring layer difference are ineffective for 12-layer encoder models. If InfoNCE increases contradiction rate for RoBERTa only, despite Cosine and CKA improving it, this would suggest that parameterized MI estimators are counterproductive at scale and simpler parameter-free methods are preferable for larger models.

Estimated GPU Hours: 1-2 hours on Colab T4 to test the 1 additional method (MITR-InfoNCE) on RoBERTa and BERT on 3 epochs.

How this changes the paper's central claim: If MITR-InfoNCE improves logical consistency on both BERT and RoBERTa, the paper can claim that MITR is effective across backbones when using a sufficiently expressive MI estimator which addresses the biggest gap in the current submission.


2.
Backbone Generalization Study
Hypothesis: If we apply MITR-CKA and MITR-Cosine to DeBERTa and ALBERT on BoolQ, then both models will show reduced contradiction rates compared to their baselines, because their pretraining differs from BERT's MLM+NSP objective, which we hypothesize is responsible for BERT's negative result.
Expected result: Both DeBERTa and ALBERT show lower contradiction rates than their baselines, consistent with RoBERTa's results, confirming BERT's NSP pretraining as the cause of its negative result.
What a negative result means: If both DeBERTa and ALBERT show increased contradiction rates, this suggests RoBERTa is the outlier and MITR-CKA and MITR-Cosine generally hurt logical consistency on 12-layer encoder models. If results are mixed meaning one model improves and one worsens, this indicates the effect is model-specific and no general conclusion can be drawn without testing additional backbones.

GPU hours: 4-6 hours on Colab T4 for 2 backbones x 2 strategies plus baselines at 5 epochs each.

How this changes the central claim: If DeBERTa and ALBERT improve, the paper can reframe BERT's negative result as a known exception tied to NSP pretraining rather than a fundamental weakness of MITR.

Ranking: Experiment 1 is ranked first because it addresses a greater weakness of the original experiments. Concluding InfoNCE is the most effective method in the initial experiment and then excluding it from larger model experiments without justification is a direct rejection risk. Experiment 2 improves the validity of the central claim by determining whether BERT's negative result is an outlier, but this is secondary to first establishing whether the paper's strongest strategy generalizes to larger models.

# D

This experiment tested InfoNCE on BERT and RoBERTa to determine if InfoNCE, which showed the most improvement on DistilBERT, generalizes to the larger 12-layer models. InfoNCE achieved a contradiction rate of 0.4920 compared to its baseline of 0.4160, a 7.6% increase. This means that InfoNCE significantly worsened the logical consistency of BERT. The accuracy also dropped from 0.6940 to 0.6887 which shows another meaningful reduction in performance. These findings do not support my hypothesis. RoBERTa MITR-InfoNCE achieved a contradiction rate of 0.4800 compared to its baseline of 0.5400, a 6.00% reduction, with accuracy unchanged at 0.7027, partially supporting the hypothesis. RoBERTa improved as predicted, but BERT did not. This backbone-dependent pattern is consistent with prior Cosine and CKA experiments where BERT's contradiction rate also worsened, suggesting BERT's MLM+NSP pretraining may make it resistant to MI regularization regardless of the estimator used.



Transformer models fine-tuned for logical reasoning tasks suffer from inter-layer redundancy, where adjacent layers learn similar representations, wasting model capacity and reducing logical consistency. We propose MITR (Mutual Information Transformer Regularization), which adds a mutual information penalty between consecutive transformer layers during fine-tuning to force each layer to learn distinct representations. We extend prior MITR experiments on DistilBERT by testing InfoNCE, the strongest MI estimator from initial experiments, on BERT and RoBERTa, two backbones omitted from prior scaling experiments without justification. Our results show that MITR-InfoNCE's effectiveness is backbone-dependent: it increases BERT's contradiction rate from 0.4160 to 0.4920 (+7.60%) while reducing RoBERTa's from 0.5400 to 0.4800 (-6.00%). These findings suggest that BERT's MLM+NSP pretraining may make it resistant to MI regularization, warranting further investigation across additional encoder backbones.